# Meeting 10, Act 3: A Word Is a Vector

**CS 301, Introduction to Data Science, NJIT.** Unit II, meeting 10.

A table and an image are made of numbers from the start. A word is not. In this act your group works with vectors that stand for words, computes with them, and tests what the arithmetic of vectors finds in language.

## How this lab works

- **This is the notebook of Act 3 of meeting 10.** It starts from its own setup cell and uses nothing that was computed in the notebooks of Acts 1 and 2, except the number of your group.
- **Work in the same group.** Each member opens this notebook and runs it. First open the **File** menu and choose **Save a copy in Drive**, so that your changes are kept.
- **Run the cells in order**, from the top. To run a cell, click it and press **Shift + Enter**.
- A paragraph that begins with **Your turn** asks you to do something before you go on. In this act, most of them ask you to replace words in a code cell with words of your own and to run the cell again.
- **In Step 23** your group posts a picture on the discussion board of this meeting. One member posts for the group, and every post begins with the names of all its members. **At the end of the meeting** each member uploads the notebooks to Canvas. The last section of this notebook says how.
- **After the meeting**, **Review Quiz 10** on Canvas asks about this lab. Where the notebook says that the quiz asks about a point, read that part again before you take the quiz.

**Your turn.** Double-click this cell and replace this line with the names of the members of your group.

In [ ]:
# Setup. Run this cell first.
import io
import time
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np

# The address of the course's files for this unit, and of the folder that holds the word vectors.
SITE = "https://ikoutis.github.io/cs301-labs/unit-2/"
EMBEDDINGS = SITE + "embeddings/"


def read_array(file):
    """Read an array of numbers from the course's files."""
    return np.load(io.BytesIO(urlopen(EMBEDDINGS + file).read())).astype(np.float32)


print("Setup is done.")

## <font color="#1967d2"><b>Step 19: A word as a list of numbers</b></font>

A **word embedding** gives every word a vector: a list of m numbers. The vector of a word is also called a **word vector**. Nobody chose these numbers by hand. A program read a very large amount of text and computed the numbers so that words that occur in similar surroundings in the text receive similar vectors.

The vectors of this act are called **GloVe** vectors. They were computed at Stanford University from Wikipedia and from news text, six billion words in all. The course keeps the vectors of the 50,000 most frequent words, in two versions: with 50 numbers for each word, and with 300.

The next cell reads the version with 50 numbers. It is a matrix `E50` with 50,000 rows and 50 columns, of the same kind as the matrix `X` of Act 1: **a row is one word, and a column is one dimension**. There is one difference. A column of `X` had a name, such as a temperature. A column of `E50` has no name and no meaning by itself.

**Your turn.** Every member types the group's number in the next cell, in place of the `0`, and runs the cell. The number gives your group a word to start from.

In [ ]:
GROUP = 0        # replace 0 with your group's number, from 1 to 30

assert 1 <= GROUP <= 30, "Type your group's number in place of the 0."

words = urlopen(EMBEDDINGS + "words.txt").read().decode().split()     # the 50,000 words, most frequent first
position = {word: k for k, word in enumerate(words)}                  # the row number of every word
E50 = read_array("glove-50.npy")

STARTERS = ["river", "guitar", "doctor", "coffee", "winter", "soccer", "piano", "bridge", "forest", "teacher",
            "ocean", "bread", "train", "garden", "mountain", "engine", "painter", "kitchen", "market", "planet",
            "island", "violin", "tiger", "hospital", "library", "airport", "chocolate", "hurricane", "bicycle", "volcano"]
WORD = STARTERS[GROUP - 1]

print(f"E50 has shape {E50.shape}: {E50.shape[0]:,} words, and {E50.shape[1]} numbers for each word")
print("the first ten words:", words[:10])
print(f"Your group's word is '{WORD}'. It is in row {position[WORD]:,}, and its vector is:")
print(np.round(E50[position[WORD]], 2))

## <font color="#1967d2"><b>Step 20: Which words are close to a word?</b></font>

The 50 numbers of one word say nothing when they are read one by one. What carries information is how the vectors of two words compare.

Two vectors are compared by the **angle** between them. The measure is the **cosine similarity**: divide each vector by its length, so that both have length 1, and take their dot product. The dot product is the sum of the products of matching entries, the same operation as the score $w \cdot x$ of a unit in Unit I. The cosine similarity is a number from −1 to 1: it is 1 when the two vectors point in the same direction, and 0 when they are perpendicular.

The next cell needs two operations on whole arrays:

- `unit_rows(E)` divides every row of the matrix by its length. `np.linalg.norm(E, axis=1, keepdims=True)` computes the 50,000 lengths, one for each row, and the division uses broadcasting, as `X - mu` did in Act 1.
- `U @ v` computes the dot product of **every row** of `U` with the vector `v`: 50,000 dot products in one operation. The symbol `@` is NumPy's product of matrices and vectors. The result holds the similarity of one word to all 50,000 words.

In [ ]:
def unit_rows(E):
    """Divide every row of E by its length, so that every row has length 1."""
    lengths = np.linalg.norm(E, axis=1, keepdims=True)      # one length for each row
    return E / lengths


def row(word):
    """The row number of a word."""
    assert word in position, f"'{word}' is not among the {len(words):,} words. All the words are in lower case."
    return position[word]


def nearest(word, U, k=8):
    """The k words whose vectors point most nearly in the direction of the vector of word."""
    similarity = U @ U[row(word)]                  # one dot product for every word
    order = np.argsort(-similarity)                # row numbers, from the most similar word on
    return [(words[i], round(float(similarity[i]), 2)) for i in order[1:k + 1]]    # order[0] is the word itself


U50 = unit_rows(E50)
print(f"The words closest to '{WORD}', with their cosine similarity:")
for other, similarity in nearest(WORD, U50):
    print(f"  {other:14s}{similarity:6.2f}")

**Your turn.** In the next cell, replace the three words with words of your own, in lower case, and run the cell. Try a word that has more than one meaning.

In [ ]:
for word in ["newark", "python", "happy"]:
    print(word, "->", [other for other, similarity in nearest(word, U50)])

**How to read the result.** A word has one vector, whatever its number of meanings. The words closest to `python` are a mixture: programming languages, animals, and a group of comedians.

## <font color="#1967d2"><b>Step 21: Arithmetic on words</b></font>

Vectors can be added and subtracted, and so can the vectors of words. The best-known example is

$$\text{king} - \text{man} + \text{woman} \approx \text{queen}.$$

In words: take the vector of *king*, subtract the vector of *man*, add the vector of *woman*. The result is a new vector, and the word whose vector is closest to it is *queen*. The reason is that the difference between the vectors of *king* and *man* is close to the difference between the vectors of *queen* and *woman*.

The function `analogy(a, b, c, U)` answers the question "a is to b as c is to which word?". It computes the vector of b, minus the vector of a, plus the vector of c, and returns the words that are closest to the result.

One detail matters. The word closest to the result is often one of the three words of the question itself. The function therefore leaves those three words out of its answer. The second line that the next cell prints shows what happens when it does not.

In [ ]:
def analogy(a, b, c, U, k=3, leave_out=True):
    """a is to b as c is to which word? The k words closest to: vector of b - vector of a + vector of c."""
    target = U[row(b)] - U[row(a)] + U[row(c)]
    similarity = U @ (target / np.linalg.norm(target))
    if leave_out:
        similarity[[row(a), row(b), row(c)]] = -2      # the three words of the question cannot be the answer
    return [words[i] for i in np.argsort(-similarity)[:k]]


print("man : king = woman : ?      ", analogy("man", "king", "woman", U50))
print("the same, nothing left out: ", analogy("man", "king", "woman", U50, leave_out=False))
print("france : paris = italy : ?  ", analogy("france", "paris", "italy", U50))
print("walk : walked = play : ?    ", analogy("walk", "walked", "play", U50))
print("cold : colder = warm : ?    ", analogy("cold", "colder", "warm", U50))

**Your turn.** The next cell asks three more questions. Run it, then replace them with three questions of your own and run it again. Find one question that the vectors answer well and one that they answer badly, and write both in the cell marked **Our questions**.

In [ ]:
print("japan : tokyo = egypt : ?   ", analogy("japan", "tokyo", "egypt", U50))
print("big : biggest = small : ?   ", analogy("big", "biggest", "small", U50))
print("mouse : mice = foot : ?     ", analogy("mouse", "mice", "foot", U50))

**Our questions.** Double-click this cell and replace this line with one question that was answered well and one that was answered badly, each with the answer that the function gave.

## <font color="#1967d2"><b>Step 22: How many numbers for each word?</b></font>

The number m of numbers that each word receives is the **dimension** of the embedding. It is chosen before the vectors are computed. With more numbers for each word, the vectors can record more distinctions between words. The price is the one that Act 1 measured: a matrix with n rows and m columns holds n × m numbers, and every operation on it visits all of them.

The next cell reads the vectors with 300 numbers for each word, and puts the same 40 questions to both versions. The questions are of six kinds: a country and its capital, a man and a woman, the forms of an adjective, the forms of a verb, one and many, and a country with its language or its money.

In [ ]:
E300 = read_array("glove-300.npy")
U300 = unit_rows(E300)
print(f"E50  has shape {E50.shape}: {E50.size:,} numbers, {E50.nbytes / 1e6:.0f} megabytes")
print(f"E300 has shape {E300.shape}: {E300.size:,} numbers, {E300.nbytes / 1e6:.0f} megabytes")

# Each line of four words reads: the first is to the second as the third is to the fourth.
QUESTIONS = [
    ("france", "paris", "italy", "rome"), ("france", "paris", "japan", "tokyo"), ("germany", "berlin", "spain", "madrid"),
    ("russia", "moscow", "egypt", "cairo"), ("greece", "athens", "poland", "warsaw"), ("china", "beijing", "ireland", "dublin"),
    ("england", "london", "austria", "vienna"), ("sweden", "stockholm", "norway", "oslo"), ("turkey", "ankara", "cuba", "havana"),
    ("canada", "ottawa", "peru", "lima"),
    ("man", "woman", "king", "queen"), ("man", "woman", "brother", "sister"), ("man", "woman", "uncle", "aunt"),
    ("man", "woman", "father", "mother"), ("man", "woman", "son", "daughter"), ("man", "woman", "actor", "actress"),
    ("boy", "girl", "prince", "princess"), ("he", "she", "his", "her"),
    ("big", "bigger", "small", "smaller"), ("good", "better", "bad", "worse"), ("fast", "faster", "slow", "slower"),
    ("strong", "stronger", "weak", "weaker"), ("big", "biggest", "small", "smallest"), ("cold", "colder", "warm", "warmer"),
    ("long", "longer", "short", "shorter"), ("high", "highest", "low", "lowest"),
    ("walk", "walked", "play", "played"), ("go", "went", "take", "took"), ("see", "saw", "give", "gave"),
    ("run", "running", "swim", "swimming"), ("eat", "ate", "write", "wrote"), ("sing", "sang", "speak", "spoke"),
    ("car", "cars", "dog", "dogs"), ("child", "children", "man", "men"), ("city", "cities", "country", "countries"),
    ("mouse", "mice", "foot", "feet"),
    ("france", "french", "spain", "spanish"), ("japan", "japanese", "germany", "german"), ("italy", "italian", "greece", "greek"),
    ("japan", "yen", "russia", "ruble"),
]

answers50 = [analogy(a, b, c, U50, k=1)[0] for a, b, c, d in QUESTIONS]
answers300 = [analogy(a, b, c, U300, k=1)[0] for a, b, c, d in QUESTIONS]
correct50 = sum(answer == d for answer, (a, b, c, d) in zip(answers50, QUESTIONS))
correct300 = sum(answer == d for answer, (a, b, c, d) in zip(answers300, QUESTIONS))

print()
print(f"Correct answers out of {len(QUESTIONS)}: with 50 numbers for each word {correct50}, with 300 numbers {correct300}")
print()
print("The questions on which the two versions give different answers:")
print(f"  {'question':38s}{'expected':12s}{'50 numbers':14s}{'300 numbers':14s}")
for (a, b, c, d), small, large in zip(QUESTIONS, answers50, answers300):
    if small != large:
        print(f"  {a + ' : ' + b + ' = ' + c + ' : ?':38s}{d:12s}{small:14s}{large:14s}")

print()
for name, U in (("50 numbers", U50), ("300 numbers", U300)):
    v = U[row(WORD)]
    start = time.perf_counter()
    for _ in range(20):
        U @ v
    seconds = (time.perf_counter() - start) / 20
    print(f"{name:12s}: comparing one word with all 50,000 words takes {seconds * 1000:.2f} milliseconds")

**How to read the result.** Compare the two counts, and then the questions on which the two versions disagree. More numbers for each word do not make every answer right: look for a question that the version with 300 numbers gets wrong. Then compare the sizes and the times.

**Your turn.** Put your group's word, and your own questions of Step 21, to the vectors with 300 numbers. In the next cell, replace the two questions with yours. Review Quiz 10 asks what changed.

In [ ]:
print(f"closest to '{WORD}', 50 numbers: ", [other for other, similarity in nearest(WORD, U50)])
print(f"closest to '{WORD}', 300 numbers:", [other for other, similarity in nearest(WORD, U300)])
print("big : biggest = small : ?   ", analogy("big", "biggest", "small", U300))
print("mouse : mice = foot : ?     ", analogy("mouse", "mice", "foot", U300))

## <font color="#1967d2"><b>Step 23: A picture of word vectors</b></font>

A vector of 300 numbers cannot be drawn. For a small set of words, a picture is still possible: find the **two directions** along which the chosen vectors are most spread out, and draw each word at its two coordinates along those directions. `np.linalg.svd` finds the directions. The picture keeps only two of the 300 dimensions, so distances in it are approximate.

The next cell draws countries and their capitals, with an arrow from each country to its capital. If the difference between a country and its capital is nearly the same vector for every pair, the arrows are nearly parallel. Before the picture, the cell prints that as numbers: for every pair, the cosine similarity between its arrow and the average of the other arrows, computed with all 300 numbers.

In [ ]:
def picture(pairs, U):
    """Draw the vectors of the words of the pairs in two dimensions, with an arrow from the first word of a pair to the second."""
    chosen = [word for pair in pairs for word in pair]
    V = U[[row(word) for word in chosen]]
    V = V - V.mean(axis=0)
    directions = np.linalg.svd(V, full_matrices=False)[2][:2]     # the two directions of greatest spread
    xy = V @ directions.T                                         # two coordinates for each word
    place = dict(zip(chosen, xy))

    arrows = np.array([U[row(b)] - U[row(a)] for a, b in pairs])
    print("Cosine similarity of each arrow with the average of the other arrows (1 means parallel):")
    for k, (a, b) in enumerate(pairs):
        others = np.delete(arrows, k, axis=0).mean(axis=0)
        cosine = arrows[k] @ others / np.linalg.norm(arrows[k]) / np.linalg.norm(others)
        print(f"  {a:12s}-> {b:12s}{cosine:6.2f}")

    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    for a, b in pairs:
        ax.annotate("", xy=place[b], xytext=place[a], arrowprops=dict(arrowstyle="->", color="#8a8a86"))
        ax.annotate(a, place[a], textcoords="offset points", xytext=(6, 5))
        ax.annotate(b, place[b], textcoords="offset points", xytext=(6, 5))
    firsts = np.array([place[a] for a, b in pairs])
    seconds = np.array([place[b] for a, b in pairs])
    ax.plot(firsts[:, 0], firsts[:, 1], "o", color="#2a78d6", markersize=7, label="first word of a pair")
    ax.plot(seconds[:, 0], seconds[:, 1], "s", color="#1a1a19", markersize=6, label="second word of a pair")
    ax.set_title(f"{len(chosen)} word vectors of {U.shape[1]} numbers, drawn in two dimensions")
    ax.set_xlabel("first direction of greatest spread")
    ax.set_ylabel("second direction of greatest spread")
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend()
    fig.tight_layout()
    plt.show()


PAIRS = [("france", "paris"), ("italy", "rome"), ("japan", "tokyo"), ("germany", "berlin"),
         ("russia", "moscow"), ("egypt", "cairo"), ("greece", "athens"), ("china", "beijing")]
picture(PAIRS, U300)

**Your turn.** Make a picture of your own. In the next cell, replace the pairs with five to eight pairs of words that share one relation: for example a verb and its past form, an animal and its young, a country and its language, or a relation that your group thinks of. Run the cell.

Then **post the picture now** on the discussion board of this meeting, as a reply to your group's first post. One member posts for the group, and it need not be the member who posted before. Begin the post with the names of all the members of your group. Add to the picture:

- one sentence that says what the picture shows, so that a classmate who cannot see it still gets the result;
- the relation you chose, and whether the arrows came out nearly parallel;
- your best question of Step 21, with its answer.

Keep the pairs and the picture. **Review Quiz 10** asks about them.

In [ ]:
OUR_PAIRS = [("walk", "walked"), ("play", "played"), ("give", "gave"), ("take", "took"), ("write", "wrote"), ("sing", "sang")]
picture(OUR_PAIRS, U300)

## <font color="#1967d2"><b>Step 24: A text is a sequence of vectors</b></font>

A text is words in an order. When each word is replaced by its vector, the text becomes a **sequence of vectors**: a matrix with one row for each word, in the order of the text. A text of L words gives a matrix with L rows and m columns.

In the table of Act 1 the order of the rows carried no information. Here it does: the rows are in the order of the words.

A text is also **one point** of a dataset, as one image was in Act 2, and it can carry a label, such as its language or its subject.

The next cell turns a sentence into its matrix and draws the matrix as a picture: one row for each word, one column for each of the 50 dimensions, and a colour for each value. A matrix drawn in this way is the same kind of object as the grey image of Act 2.

In [ ]:
def text_matrix(text, E):
    """The matrix of a text: one row for each word, in the order of the text."""
    return E[[row(word) for word in text.lower().split()]]


SENTENCE = "the river flows through the valley to the sea"       # your turn: a sentence of your own, without punctuation
S = text_matrix(SENTENCE, E50)
print(f"'{SENTENCE}'")
print(f"shape of S: {S.shape}, which is {S.shape[0]} words, and {S.shape[1]} numbers for each word")

largest = float(np.abs(S).max())
fig, ax = plt.subplots(figsize=(10, 0.36 * len(S) + 1.3))
drawing = ax.imshow(S, cmap="coolwarm", vmin=-largest, vmax=largest, aspect="auto")
ax.set_yticks(range(len(S)))
ax.set_yticklabels(SENTENCE.lower().split())
ax.set_xlabel("dimension of the embedding, from 0 to 49")
ax.set_title("The matrix of one sentence: one row for each word")
fig.colorbar(drawing, ax=ax, label="value (blue: negative, red: positive)")
fig.tight_layout()
plt.show()

first = text_matrix("the dog chased the cat", E50)
second = text_matrix("the cat chased the dog", E50)
print("'the dog chased the cat' and 'the cat chased the dog':")
print("   rows 0 and 3 of the first matrix, both the word 'the', are equal:", np.array_equal(first[0], first[3]))
print("   the two matrices are equal:", np.array_equal(first, second))
print("   the averages of their rows are equal:", np.allclose(first.mean(axis=0), second.mean(axis=0)))

**How to read the result.** A word has the same row wherever it occurs: in the picture, the rows of the word *the* are alike. The two sentences about the dog and the cat have the same words, and therefore the same rows, in a different order, so their matrices differ. An operation that ignores the order of the rows, such as their average, cannot tell the two sentences apart.

**Your turn.** In the cell above, replace the sentence with one of your own, in lower-case words and without punctuation, and run the cell again.

## <font color="#1967d2"><b>Step 25: A batch of texts</b></font>

Several texts are stored together as a **batch**, like the images of Act 2. One difficulty is new. Texts have different lengths, and an array needs the same number of rows for every text. The usual remedy is **padding**: every text that is shorter than the longest one is filled up, at its end, with rows of zeros.

A batch of B texts, padded to L words, with m numbers for each word, is an array with **three** dimensions:

| Data | One point | A batch |
|---|---|---|
| A row of a table (Act 1) | m numbers | n by m |
| A colour image (Act 2) | H by W by 3 | n by H by W by 3 |
| A text of L words | L by m | B by L by m |

Programs that process language, large language models among them, receive text in this form: a batch of sequences of vectors.

In [ ]:
def text_batch(texts, E):
    """Stack the matrices of several texts into one array, padded with rows of zeros to the longest text."""
    matrices = [text_matrix(text, E) for text in texts]
    longest = max(len(M) for M in matrices)
    batch = np.zeros((len(texts), longest, E.shape[1]), dtype=E.dtype)
    for k, M in enumerate(matrices):
        batch[k, :len(M)] = M                    # the rows after the last word stay zero
    return batch


TEXTS = ["the river flows through the valley to the sea",          # your turn: three sentences of your own
         "students write code",
         "she scored two goals in the final match"]
T = text_batch(TEXTS, E50)
print("shape of T:", T.shape, "- text, position in the text, dimension of the embedding")
for k, text in enumerate(TEXTS):
    count = len(text.split())
    print(f"   text {k}: {count} words and {T.shape[1] - count} rows of zeros: '{text}'")
print(f"numbers in the batch: {T.size:,}")

**Your turn.** In the cell above, replace the three texts with three sentences of your own, one from each member of the group, and run the cell again. Note the shape of your batch: Review Quiz 10 asks how such a shape comes about.

## At the end of the meeting: hand in

**Your group has posted three times** on the discussion board of this meeting: at Step 9 in Act 1, at Step 15 in Act 2, and at Step 23 in Act 3. If one of the three posts is missing, add it now. Check that every post names all the members of your group.

**Read the posts of two other groups.** Their datasets have other sizes: compare their nanoseconds per number with yours.

**Each member hands in the three notebooks.** In the notebooks of Acts 1, 2 and 3, open the **File** menu, choose **Download**, then **Download .ipynb**. Upload the three files to this meeting's class-work assignment on Canvas.

**After the meeting**, take **Review Quiz 10** on Canvas. Some of its questions ask for an answer of a few sentences. Your notebooks, with your group's measurements, prompts, questions and pictures in them, are the reference for them.